In [2]:
# Carga Excel -> UbicacionLote (idempotente, sin duplicados)
import os, sys
sys.path.insert(0, '../')
sys.path.insert(0, '../../src')
from dotenv import load_dotenv
import pandas as pd
import sqlalchemy as sa
from sqlalchemy.engine import URL
from upload_warehouse_excel.load_ubicacion import (
    prepare_excel, load_lote, canonical_lote, resolve, upsert, verify_no_duplicates,
)
load_dotenv()
print('imports OK')

imports OK


In [3]:
def connect_to_database():
    req = {k: os.getenv(k) for k in ['DB_SERVER', 'DB_NAME', 'DB_USER', 'DB_PASSWORD']}
    missing = [k for k, v in req.items() if not v]
    if missing:
        raise RuntimeError(f'Faltan variables de entorno: {missing}')
    url = URL.create(
        'mssql+pyodbc', username=req['DB_USER'], password=req['DB_PASSWORD'],
        host=req['DB_SERVER'], port=int(os.getenv('DB_PORT', '1433')), database=req['DB_NAME'],
        query={'driver': os.getenv('DB_DRIVER', 'ODBC Driver 18 for SQL Server'),
                 'Encrypt': os.getenv('DB_ENCRYPT', 'no'),
                 'TrustServerCertificate': os.getenv('DB_TRUST_SERVER_CERT', 'yes')},
    )
    return sa.create_engine(url, pool_pre_ping=True, pool_recycle=1800,
                            connect_args={'fast_executemany': True})  # inserts por lotes, evita que se quede pegado

conn = connect_to_database()
print('engine OK')

engine OK


In [4]:
# 1. Excel: normaliza + agrega por (imp, pallet, codigo, lote, ubicacion) sumando Cant.
# NULL de UBICACION = pendiente de migrar (se conserva, no se filtra).
df_total = pd.read_excel('../../data.xlsx', dtype={'CODIGO': str, 'LOTE': str})
df_excel_agg, stats_excel = prepare_excel(df_total)
print(stats_excel)
df_excel_agg.head()

{'excel_total': 3050, 'excel_sin_ubicacion': 1019, 'excel_cant_nula': 0, 'excel_imp_nulo': 0, 'excel_sin_llave': 1, 'excel_grupos': 3030}


,imp_k,pallet_k,codigo_k,lote_k,ubicacion_k,cantidad,lineas,ubicacion_rep,pallet_rep
0,001-26 TOUS,P3 FRAGANCIA,04-731061,M13085,R01-32-05,154,1,R01-32-05,P3 FRAGANCIA
1,001-26 TOUS,P4 FRAGANCIA,04-5004000,M21664,R05-18-06,306,1,R05-18-06,P4 FRAGANCIA
2,004-26 TED LAPIDUS,P1 FRAGANCIA,231017,25G09,R04-07-05,42,1,R04-07-05,P1 FRAGANCIA
3,004-26 TED LAPIDUS,P1 FRAGANCIA,235017,25F22,R04-07-05,150,1,R04-07-05,P1 FRAGANCIA
4,004-26 TED LAPIDUS,P2 FRAGANCIA,230017,24J12,R04-11-05,1100,1,R04-11-05,P2 FRAGANCIA


In [5]:
# 2. Lote (SOLO lectura de otras tablas; JOIN por referenciaEntradaId).
# Puente de 4 campos; 1 id canonico MIN(id) por llave.
df_lote = load_lote(conn)
df_canon, df_dup_origen = canonical_lote(df_lote)
print(f'lote filas={len(df_lote)} canon={len(df_canon)} llaves_con_multi_id={len(df_dup_origen)}')
df_dup_origen.head()

lote filas=48670 canon=47299 llaves_con_multi_id=1307


,imp_k,pallet_k,codigo_k,lote_k,canon_id,n_ids
43,001-24 COTY MAX FACTOR,P2 MAQUILLAJE,91035668,2088,2382,2
85,001-26 TOUS,P3 FRAGANCIA,04-4904000,M12515,37170,2
86,001-26 TOUS,P3 FRAGANCIA,04-4904000,M33075,37171,2
113,002-24 LVMH,P1 FRAGANCIA,K85188000,3F01,2009,2
182,002-24 LVMH,P1 MAQUILLAJE,P090456,3J02,2128,2


In [6]:
# 3. Cruce INNER por 4 campos + verificacion 0 duplicados en (lote, ubicacion).
df_cand, df_missing = resolve(df_excel_agg, df_canon)
print(verify_no_duplicates(df_cand))
print(f'match={len(df_cand)}  excel_sin_match={len(df_missing)} (van a reporte, NO se insertan)')
df_missing.head()

{'filas': 2360, 'dup_clave': 0, 'nulos_ubicacion': 1003}
match=2360  excel_sin_match=670 (van a reporte, NO se insertan)


,imp_k,pallet_k,codigo_k,lote_k,ubicacion_k,cantidad,lineas,ubicacion_rep,pallet_rep,canon_id,n_ids
7,006-26 NORTHCASTLE,P1 FRAGANCIA,LE961600,38Z200B,R03-22-05,40,1,R03-22-05,P1 FRAGANCIA,NaN,NaN
8,006-26 NORTHCASTLE,P1 FRAGANCIA,LF441300,38Z503P,R03-22-05,48,1,R03-22-05,P1 FRAGANCIA,NaN,NaN
9,006-26 NORTHCASTLE,P3 FRAGANCIA,L7847900,38Z61VJ,R04-05-06,15,1,R04-05-06,P3 FRAGANCIA,NaN,NaN
10,006-26 NORTHCASTLE,P3 FRAGANCIA,LE137900,22Z11D6,R04-05-06,18,1,R04-05-06,P3 FRAGANCIA,NaN,NaN
11,006-26 NORTHCASTLE,P3 FRAGANCIA,LF123401,22Z41AF,R04-05-06,270,1,R04-05-06,P3 FRAGANCIA,NaN,NaN


In [7]:
# 4. Candidato a UbicacionLote (entradaloteid, nroPallete, cantidad, ubicacion).
df_cand.head(10)
print('pendientes NULL:', int(df_cand['ubicacion'].isna().sum()))

pendientes NULL: 1003


In [ ]:
# 5. Carga idempotente: SOLO escribe UbicacionLote.
# Recarga logica por entradaloteid afectado (soft-delete + insert en transaccion).
# Re-ejecutar NO duplica: el UNIQUE (entradaloteid, ubicacion_key) lo blinda.
res = upsert(conn, df_cand)
print(res)

In [ ]:
# 6. Verificacion post-carga.
with conn.connect() as c:
    tot = pd.read_sql("SELECT COUNT(*) AS activos FROM dbo.UbicacionLote WHERE estado=1", c)
    dup = pd.read_sql("""SELECT COUNT(*) AS grupos_dup FROM (SELECT entradaloteid FROM dbo.UbicacionLote WHERE estado=1 GROUP BY entradaloteid, COALESCE(UPPER(LTRIM(RTRIM(ubicacion))), '##SIN_UBICAR##') HAVING COUNT(*)>1) t""", c)
print(tot.to_dict('records'), dup.to_dict('records'))
assert int(dup.iloc[0, 0]) == 0, 'Hay duplicados en (entradaloteid, ubicacion)'
print('verificacion OK: 0 duplicados')